# Span-Level F-Score Analysis: Best Model vs Human Annotations

This notebook calculates span-level F-scores (precision, recall, F1) comparing the **best-performing model** (70B Zero-Shot) with human span annotations.

## Best Model:
- **70B Zero-Shot** (F1=0.6296 on conspiracy classification)

## Analysis:
- Compares model-extracted spans with human-annotated spans
- Calculates metrics for each span element type: plan, secret, in-group, out-group, call-to-action
- Uses character-level overlap for matching spans
- Generates comprehensive span-level performance table

## Output:
- Span-level F-score table (precision, recall, F1 for each span type)
- CSV export for paper inclusion
- LaTeX table format for paper inclusion


In [15]:
# Setup & Imports
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Set paths
BASE = Path('/home/ashrafs/projects/Conspiracy-Proj')
DATA = BASE / 'Data'
OUT = BASE / 'outputs'
PUB_TABLES = BASE / 'publication' / 'tables'
PUB_TABLES.mkdir(exist_ok=True, parents=True)

# Best model: 70B Zero-Shot (based on classification F1=0.6296)
BEST_MODEL_NAME = '70B Zero-Shot'
BEST_MODEL_SPAN_FILE = OUT / 'llama-3.3-70B-zeroshot_all_span_annotations.csv'

print("✓ Setup complete!")
print(f"Data directory: {DATA}")
print(f"Output directory: {OUT}")
print(f"Tables directory: {PUB_TABLES}")
print(f"\nBest Model: {BEST_MODEL_NAME}")
print(f"Model span file: {BEST_MODEL_SPAN_FILE.name}")


✓ Setup complete!
Data directory: /home/ashrafs/projects/Conspiracy-Proj/Data
Output directory: /home/ashrafs/projects/Conspiracy-Proj/outputs
Tables directory: /home/ashrafs/projects/Conspiracy-Proj/publication/tables

Best Model: 70B Zero-Shot
Model span file: llama-3.3-70B-zeroshot_all_span_annotations.csv


## 1. Helper Functions for Span Comparison


In [16]:
# Helper Functions for Span Text Comparison
def normalize_span_text(text):
    """Normalize span text for comparison."""
    if pd.isna(text) or text == '':
        return ''
    # Convert to string, lowercase, strip whitespace
    text = str(text).strip().lower()
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text

def split_spans(text):
    """Split span text by || delimiter and normalize."""
    if pd.isna(text) or text == '':
        return []
    spans = str(text).split('||')
    return [normalize_span_text(s.strip()) for s in spans if normalize_span_text(s.strip())]

def compute_character_overlap(text1, text2):
    """Compute character-level overlap metrics between two texts."""
    text1 = normalize_span_text(text1)
    text2 = normalize_span_text(text2)
    
    if not text1 or not text2:
        return 0.0, 0.0, 0.0, 0.0  # intersection, text1_len, text2_len, overlap_ratio
    
    # Character sets (multiset - count characters)
    chars1 = Counter(text1)
    chars2 = Counter(text2)
    
    intersection = sum((chars1 & chars2).values())
    union = sum((chars1 | chars2).values())
    
    # Jaccard similarity (intersection over union)
    overlap_ratio = intersection / union if union > 0 else 0.0
    
    return intersection, len(text1), len(text2), overlap_ratio

def compute_span_f1(human_spans_list, model_spans_list, overlap_threshold=0.5):
    """
    Compute F1, precision, and recall for spans using character-level overlap.
    
    Args:
        human_spans_list: list of human span texts (may be pipe-separated)
        model_spans_list: list of model span texts (may be pipe-separated)
        overlap_threshold: minimum overlap ratio to count as a match
    
    Returns:
        dict with precision, recall, f1
    """
    # Split and flatten all spans
    human_spans = []
    for span_text in human_spans_list:
        if pd.notna(span_text) and str(span_text).strip():
            human_spans.extend(split_spans(span_text))
    
    model_spans = []
    for span_text in model_spans_list:
        if pd.notna(span_text) and str(span_text).strip():
            model_spans.extend(split_spans(span_text))
    
    if not human_spans and not model_spans:
        return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'n_human': 0, 'n_model': 0, 'n_matched': 0}
    
    if not human_spans:
        return {'precision': 0.0, 'recall': 1.0, 'f1': 0.0, 'n_human': 0, 'n_model': len(model_spans), 'n_matched': 0}
    
    if not model_spans:
        return {'precision': 1.0, 'recall': 0.0, 'f1': 0.0, 'n_human': len(human_spans), 'n_model': 0, 'n_matched': 0}
    
    # Match model spans to human spans using character overlap
    matched_human = set()
    matched_pairs = []
    
    for model_span in model_spans:
        best_match_idx = None
        best_overlap = 0.0
        
        for i, human_span in enumerate(human_spans):
            if i in matched_human:
                continue
            
            _, _, _, overlap_ratio = compute_character_overlap(model_span, human_span)
            
            if overlap_ratio > best_overlap and overlap_ratio >= overlap_threshold:
                best_overlap = overlap_ratio
                best_match_idx = i
        
        if best_match_idx is not None:
            matched_human.add(best_match_idx)
            matched_pairs.append((model_span, human_spans[best_match_idx]))
    
    n_matched = len(matched_pairs)
    n_model = len(model_spans)
    n_human = len(human_spans)
    
    precision = n_matched / n_model if n_model > 0 else 0.0
    recall = n_matched / n_human if n_human > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'n_human': n_human,
        'n_model': n_model,
        'n_matched': n_matched
    }

print("✓ Helper functions defined")


✓ Helper functions defined


## 2. Load Human and Model Span Annotations


In [17]:
# Load Human Span Annotations
human_span_path = DATA / 'all_span_annotations.csv'
if not human_span_path.exists():
    raise FileNotFoundError(f"Human span file not found: {human_span_path}")

human_spans = pd.read_csv(human_span_path)
print(f"✓ Loaded human spans: {len(human_spans):,} rows")

# Load Best Model Span Annotations
if not BEST_MODEL_SPAN_FILE.exists():
    raise FileNotFoundError(f"Model span file not found: {BEST_MODEL_SPAN_FILE}")

model_spans = pd.read_csv(BEST_MODEL_SPAN_FILE)
print(f"✓ Loaded model spans ({BEST_MODEL_NAME}): {len(model_spans):,} rows")

# Define span element types
SPAN_TYPES = ['plan', 'secret', 'in-group', 'out-group', 'call-to-action']
MANDATORY_SPANS = ['plan', 'secret']
OPTIONAL_SPANS = ['in-group', 'out-group', 'call-to-action']
DESIRED_ORDER = MANDATORY_SPANS + OPTIONAL_SPANS  # Order for table display: mandatory first, then optional

print(f"\nSpan element types: {SPAN_TYPES}")
print(f"Mandatory: {MANDATORY_SPANS}")
print(f"Optional: {OPTIONAL_SPANS}")
print(f"Unique texts in human spans: {human_spans['text_id'].nunique():,}")
print(f"Unique texts in model spans: {model_spans['text_id'].nunique():,}")


✓ Loaded human spans: 934 rows
✓ Loaded model spans (70B Zero-Shot): 2,077 rows

Span element types: ['plan', 'secret', 'in-group', 'out-group', 'call-to-action']
Mandatory: ['plan', 'secret']
Optional: ['in-group', 'out-group', 'call-to-action']
Unique texts in human spans: 755
Unique texts in model spans: 2,077


## 3. Calculate Span-Level F-Scores


In [18]:
# Find texts that exist in both human and model annotations
common_text_ids = set(human_spans['text_id'].unique()) & set(model_spans['text_id'].unique())
print(f"✓ Common texts with both human and model annotations: {len(common_text_ids):,}")

# Filter to common texts
human_spans_filtered = human_spans[human_spans['text_id'].isin(common_text_ids)].copy()
model_spans_filtered = model_spans[model_spans['text_id'].isin(common_text_ids)].copy()

# Calculate F-scores for each span type
results = []

for span_type in SPAN_TYPES:
    if span_type not in human_spans.columns or span_type not in model_spans.columns:
        print(f"⚠ Warning: {span_type} not found in spans, skipping...")
        continue
    
    # Get span texts for this type from filtered dataframes
    human_span_texts = human_spans_filtered[span_type].dropna().tolist()
    model_span_texts = model_spans_filtered[span_type].dropna().tolist()
    
    # Calculate F1, precision, recall
    metrics = compute_span_f1(human_span_texts, model_span_texts, overlap_threshold=0.5)
    
    results.append({
        'Span_Type': span_type,
        'Precision': metrics['precision'],
        'Recall': metrics['recall'],
        'F1': metrics['f1'],
        'N_Human_Spans': metrics['n_human'],
        'N_Model_Spans': metrics['n_model'],
        'N_Matched': metrics['n_matched']
    })
    
    print(f"  {span_type:15s}: F1={metrics['f1']:.4f}, P={metrics['precision']:.4f}, R={metrics['recall']:.4f}")

span_metrics_df = pd.DataFrame(results)
print(f"\n✓ Calculated metrics for {len(span_metrics_df)} span types")


✓ Common texts with both human and model annotations: 749
  plan           : F1=0.6995, P=0.8907, R=0.5758
  secret         : F1=0.4866, P=0.7692, R=0.3559
  in-group       : F1=0.5714, P=0.9714, R=0.4048
  out-group      : F1=0.7774, P=0.8732, R=0.7006
  call-to-action : F1=0.6958, P=0.5621, R=0.9128

✓ Calculated metrics for 5 span types


## 4. Display Results Table


In [20]:
# Sort by custom order: mandatory elements first (plan, secret), then optional elements
# DESIRED_ORDER is already defined in cell 5

# Create a mapping for custom sort order
span_metrics_df['sort_order'] = span_metrics_df['Span_Type'].apply(
    lambda x: DESIRED_ORDER.index(x) if x in DESIRED_ORDER else 999
)

span_metrics_sorted = span_metrics_df.sort_values('sort_order').copy()
span_metrics_sorted = span_metrics_sorted.drop('sort_order', axis=1)

print("=" * 80)
print(f"SPAN-LEVEL F-SCORE ANALYSIS: {BEST_MODEL_NAME} vs Human")
print("=" * 80)
print(f"\nTask: Span Extraction (Character-level overlap matching)")
print(f"Overlap Threshold: 0.5 (Jaccard similarity)")
print(f"\n{'='*80}\n")

display(span_metrics_sorted.round(4))

# Summary statistics
print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print(f"{'='*80}")
print(f"Average F1: {span_metrics_sorted['F1'].mean():.4f}")
print(f"Average Precision: {span_metrics_sorted['Precision'].mean():.4f}")
print(f"Average Recall: {span_metrics_sorted['Recall'].mean():.4f}")
print(f"Best F1: {span_metrics_sorted['F1'].max():.4f} ({span_metrics_sorted.loc[span_metrics_sorted['F1'].idxmax(), 'Span_Type']})")
print(f"Worst F1: {span_metrics_sorted['F1'].min():.4f} ({span_metrics_sorted.loc[span_metrics_sorted['F1'].idxmin(), 'Span_Type']})")
print(f"{'='*80}")


SPAN-LEVEL F-SCORE ANALYSIS: 70B Zero-Shot vs Human

Task: Span Extraction (Character-level overlap matching)
Overlap Threshold: 0.5 (Jaccard similarity)




,Span_Type,Precision,Recall,F1,N_Human_Spans,N_Model_Spans,N_Matched
0,plan,0.8907,0.5758,0.6995,778,503,448
1,secret,0.7692,0.3559,0.4866,281,130,100
2,in-group,0.9714,0.4048,0.5714,504,210,204
3,out-group,0.8732,0.7006,0.7774,521,418,365
4,call-to-action,0.5621,0.9128,0.6958,218,354,199



SUMMARY STATISTICS
Average F1: 0.6461
Average Precision: 0.8133
Average Recall: 0.5900
Best F1: 0.7774 (out-group)
Worst F1: 0.4866 (secret)


## 5. Export Results


In [21]:
# Export as CSV
if len(span_metrics_df) > 0:
    csv_path = PUB_TABLES / 'span_fscore_best_model.csv'
    # Use the same ordered dataframe for CSV export
    span_metrics_ordered = span_metrics_df.copy()
    span_metrics_ordered['sort_order'] = span_metrics_ordered['Span_Type'].apply(
        lambda x: DESIRED_ORDER.index(x) if x in DESIRED_ORDER else 999
    )
    span_metrics_ordered = span_metrics_ordered.sort_values('sort_order').drop('sort_order', axis=1)
    
    span_metrics_ordered[['Span_Type', 'Precision', 'Recall', 'F1', 'N_Human_Spans', 'N_Model_Spans', 'N_Matched']].to_csv(
        csv_path, index=False, float_format='%.4f'
    )
    print(f"✓ Exported CSV: {csv_path}")
    
    # Export as LaTeX table
    latex_path = PUB_TABLES / 'span_fscore_best_model.tex'
    
    latex_str = "\\begin{table*}[t]\n"
    latex_str += "\\centering\n"
    latex_str += f"\\caption{{Span-Level F-Score Analysis: {BEST_MODEL_NAME} vs Human Annotations}}\n"
    latex_str += "\\label{tab:span_fscore_best_model}\n"
    latex_str += "\\begin{tabular}{lcccc}\n"
    latex_str += "\\toprule\n"
    latex_str += "Span Type & Precision & Recall & F1 \\\\\n"
    latex_str += "\\midrule\n"
    
    # Ensure the order is correct (mandatory first, then optional)
    span_metrics_ordered = span_metrics_df.copy()
    span_metrics_ordered['sort_order'] = span_metrics_ordered['Span_Type'].apply(
        lambda x: DESIRED_ORDER.index(x) if x in DESIRED_ORDER else 999
    )
    span_metrics_ordered = span_metrics_ordered.sort_values('sort_order').drop('sort_order', axis=1)
    
    for _, row in span_metrics_ordered.iterrows():
        span_type = row['Span_Type'].replace('-', '\\textendash ')  # Handle hyphens in LaTeX
        latex_str += f"{span_type} & {row['Precision']:.4f} & {row['Recall']:.4f} & {row['F1']:.4f} \\\\\n"
    
    latex_str += "\\bottomrule\n"
    latex_str += "\\end{tabular}\n"
    latex_str += "\\end{table*}\n"
    
    with open(latex_path, 'w') as f:
        f.write(latex_str)
    
    print(f"✓ Exported LaTeX table: {latex_path}")
    
    print("\n" + "=" * 80)
    print("Export Summary:")
    print(f"  - CSV: {csv_path}")
    print(f"  - LaTeX: {latex_path}")
    print("=" * 80)
else:
    print("No span metrics to export.")


✓ Exported CSV: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/span_fscore_best_model.csv
✓ Exported LaTeX table: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/span_fscore_best_model.tex

Export Summary:
  - CSV: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/span_fscore_best_model.csv
  - LaTeX: /home/ashrafs/projects/Conspiracy-Proj/publication/tables/span_fscore_best_model.tex
